In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import residue, freqz, freqs, cont2discrete
from ipywidgets import HTML
from IPython.display import display

# ============================================================
# IMPULSE-INVARIANCE BUTTERWORTH DESIGN
# Complete analytical solution and independent SciPy verification
# ============================================================

plt.ioff()

CONTENT_WIDTH = '1160px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12.5,'axes.labelsize':10.5,'xtick.labelsize':9.5,'ytick.labelsize':9.5,'legend.fontsize':8.6})

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.ii-root{
    width:1160px;
    max-width:1160px;
    font-family:Arial,sans-serif;
}

.ii-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:11px 15px;
    border-radius:8px 8px 0 0;
    font-size:18px;
    font-weight:bold;
}

.ii-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:11px 14px;
    border-radius:0 0 8px 8px;
    font-size:14px;
    line-height:1.55;
    margin-bottom:9px;
}

.ii-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:10px 12px;
    margin-bottom:8px;
    font-size:13.5px;
    line-height:1.50;
}

.ii-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:14.5px;
    margin-bottom:6px;
}

.ii-cols{
    display:flex;
    gap:14px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.ii-col{
    flex:1;
    min-width:0;
    white-space:nowrap;
}

.ii-note{
    background:#fff9e8;
    border:1px solid #d9c477;
}

.ii-ok{
    background:#eef7ee;
    border:1px solid #9cc79c;
}

.ii-code{
    text-align:center;
    font-family:Consolas,monospace;
    font-size:14px;
    margin:10px 0;
    white-space:nowrap;
}

.ii-equation{
    text-align:center;
    font-size:14px;
    line-height:1.8;
    margin:8px 0;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="ii-root">

<div class="ii-header">
Butterworth IIR Filter Design by Impulse Invariance
</div>

<div class="ii-doc">

<b>Design specifications.</b>
Construct a digital Butterworth IIR filter by the <i>impulse-invariance</i> method for

<div style="text-align:center;font-size:16px;margin:10px 0;">
<b>
ω<sub>p</sub> = 0.15π,
&nbsp;&nbsp;
ω<sub>s</sub> = 0.35π,
&nbsp;&nbsp;
A<sub>p</sub> = 3 dB,
&nbsp;&nbsp;
A<sub>s</sub> = 20 dB.
</b>
</div>

The complete design is constructed <b>step by step from the theoretical equations</b>.
The Butterworth order, cutoff frequency, analog poles, partial-fraction residues,
mapped digital poles, digital transfer function, impulse responses, and attenuation
values are obtained explicitly without using an automatic impulse-invariance conversion.

<br><br>

For simplicity, and as allowed by the theoretical development, we set
<b>T<sub>s</sub> = 1</b>.

<br><br>

Only after the complete analytical/numerical solution has been obtained, the result is
verified independently with the SciPy implementation

<div class="ii-code">
<b>scipy.signal.cont2discrete((b<sub>a</sub>, a<sub>a</sub>), T<sub>s</sub>, method='impulse')</b>
</div>

Thus, the SciPy function is used <b>only as an independent verification tool</b>
and not to obtain the solution.

</div>

</div>
"""))

# ============================================================
# SPECIFICATIONS
# ============================================================

Ts = 1.0

wp = 0.15*np.pi
ws = 0.35*np.pi

Ap = 3.0
As = 20.0

Omega_p = wp/Ts
Omega_s = ws/Ts

# ============================================================
# STEP 1 — BUTTERWORTH ORDER
# ============================================================

N_real = 0.5*np.log((10**(As/10)-1)/(10**(Ap/10)-1))/np.log(Omega_s/Omega_p)
N = int(np.ceil(N_real))

# ============================================================
# STEP 2 — CUTOFF FREQUENCY
# Passband specification is satisfied exactly
# ============================================================

Omega_c = Omega_p/(10**(Ap/10)-1)**(1/(2*N))
Omega_c_stop = Omega_s/(10**(As/10)-1)**(1/(2*N))

# ============================================================
# STEP 3 — ANALOG BUTTERWORTH POLES
# ============================================================

p1 = -Omega_c
p2 = Omega_c*(-0.5+1j*np.sqrt(3)/2)
p3 = Omega_c*(-0.5-1j*np.sqrt(3)/2)

analog_poles = np.array([p1,p2,p3])

# ============================================================
# ANALOG TRANSFER FUNCTION
#
# H_a(s) = Omega_c^3 /
#          [(s-p1)(s-p2)(s-p3)]
# ============================================================

analog_den = np.real_if_close(np.poly(analog_poles)).astype(float)
analog_num = np.array([Omega_c**N],dtype=float)

# ============================================================
# STEP 4 — PARTIAL-FRACTION EXPANSION
# ============================================================

residues,poles_pf,direct_term = residue(analog_num,analog_den)

idx_real = np.argmin(np.abs(np.imag(poles_pf)))
idx_complex = [i for i in range(len(poles_pf)) if i != idx_real]
idx_complex = sorted(idx_complex,key=lambda i: np.imag(poles_pf[i]),reverse=True)
order = [idx_real] + idx_complex

residues = residues[order]
poles_pf = poles_pf[order]

# ============================================================
# STEP 5 — IMPULSE-INVARIANCE POLE MAPPING
#
# z_k = exp(p_k Ts)
# ============================================================

digital_poles = np.exp(poles_pf*Ts)

# ============================================================
# STEP 6 — DIGITAL TRANSFER FUNCTION
#
# H(z) = sum C_k / (1 - z_k z^-1)
# ============================================================

digital_den = np.real_if_close(np.poly(digital_poles)).astype(float)
digital_num_complex = np.zeros(N,dtype=complex)

for k in range(N):
    remaining_poles = np.delete(digital_poles,k)
    digital_num_complex += residues[k]*np.poly(remaining_poles)

digital_num = np.real_if_close(digital_num_complex).astype(float)

digital_num[np.abs(digital_num)<1e-12] = 0.0
digital_den[np.abs(digital_den)<1e-12] = 0.0

# ============================================================
# FREQUENCY-RESPONSE EVALUATION
# ============================================================

def evaluate_Hz(b,a,omega):

    q = np.exp(-1j*omega)

    numerator = np.sum(b*q**np.arange(len(b)))
    denominator = np.sum(a*q**np.arange(len(a)))

    return numerator/denominator

# ============================================================
# SPECIFICATION VERIFICATION
# ============================================================

H_wp = evaluate_Hz(digital_num,digital_den,wp)
H_ws = evaluate_Hz(digital_num,digital_den,ws)

mag_wp = np.abs(H_wp)
mag_ws = np.abs(H_ws)

db_wp = 20*np.log10(mag_wp)
db_ws = 20*np.log10(mag_ws)

# ============================================================
# ANALOG IMPULSE RESPONSE
# ============================================================

t = np.linspace(0,50,5000)

h_analog = np.zeros_like(t,dtype=complex)

for Ck,pk in zip(residues,poles_pf):
    h_analog += Ck*np.exp(pk*t)

h_analog = np.real_if_close(h_analog).astype(float)

# ============================================================
# DIGITAL IMPULSE RESPONSE
#
# h[n] = h_a(n Ts)
# ============================================================

n = np.arange(0,51)

h_digital = np.zeros_like(n,dtype=complex)

for Ck,zk in zip(residues,digital_poles):
    h_digital += Ck*(zk**n)

h_digital = np.real_if_close(h_digital).astype(float)

# ============================================================
# ANALOG FREQUENCY RESPONSE
# ============================================================

Omega = np.logspace(-2,2,4000)

_,H_analog = freqs(analog_num,analog_den,worN=Omega)

Ha_dB = 20*np.log10(np.maximum(np.abs(H_analog),1e-12))

# ============================================================
# DIGITAL FREQUENCY RESPONSE
# ============================================================

omega,H_digital = freqz(digital_num,digital_den,worN=32768)

omega_normalized = omega/np.pi

Hd_dB = 20*np.log10(np.maximum(np.abs(H_digital),1e-12))

# ============================================================
# FINAL INDEPENDENT SOFTWARE VERIFICATION
#
# IMPORTANT:
# The following function is used only AFTER the complete
# impulse-invariance solution has already been obtained.
# ============================================================

scipy_num,scipy_den,scipy_dt = cont2discrete((analog_num,analog_den),Ts,method='impulse')

scipy_num = np.squeeze(scipy_num)
scipy_den = np.squeeze(scipy_den)

scipy_num[np.abs(scipy_num)<1e-12] = 0.0
scipy_den[np.abs(scipy_den)<1e-12] = 0.0

manual_num = np.pad(digital_num,(0,len(scipy_num)-len(digital_num)))
manual_den = digital_den.copy()

numerator_difference = np.max(np.abs(manual_num-scipy_num))
denominator_difference = np.max(np.abs(manual_den-scipy_den))

verification_passed = np.allclose(manual_num,scipy_num,rtol=1e-10,atol=1e-10) and np.allclose(manual_den,scipy_den,rtol=1e-10,atol=1e-10)

# ============================================================
# MAIN NUMERICAL RESULTS
# ============================================================

display(HTML(f"""
<div class="ii-root">

<div class="ii-box">

<div class="ii-title">Main numerical results</div>

<div class="ii-cols">

<div class="ii-col">
Sampling period: <b>T<sub>s</sub> = {Ts:.1f}</b><br>
Analog passband edge: <b>Ω<sub>p</sub> = {Omega_p:.6f}</b><br>
Analog stopband edge: <b>Ω<sub>s</sub> = {Omega_s:.6f}</b>
</div>

<div class="ii-col">
Calculated order: <b>N* = {N_real:.6f}</b><br>
Selected order: <b>N = {N}</b><br>
Cutoff frequency: <b>Ω<sub>c</sub> = {Omega_c:.6f}</b>
</div>

<div class="ii-col">
|H(e<sup>jωp</sup>)| = <b>{mag_wp:.6f}</b><br>
At ω<sub>p</sub>: <b>{db_wp:.6f} dB</b><br>
At ω<sub>s</sub>: <b>{db_ws:.6f} dB</b>
</div>

<div class="ii-col">
Analog impulse maximum: <b>{np.max(h_analog):.6f}</b><br>
Digital impulse maximum: <b>{np.max(h_digital):.6f}</b><br>
Digital poles stable: <b>{"YES" if np.all(np.abs(digital_poles)<1) else "NO"}</b>
</div>

</div>

</div>

</div>
"""))

# ============================================================
# ANALOG POLES AND RESIDUES
# ============================================================

display(HTML(f"""
<div class="ii-root">

<div class="ii-box">

<div class="ii-title">Analog poles and partial-fraction residues</div>

<div class="ii-cols">

<div class="ii-col">
<b>p₁ = {np.real(poles_pf[0]):.6f} {np.imag(poles_pf[0]):+.6f}j</b><br>
<b>C₁ = {np.real(residues[0]):.6f} {np.imag(residues[0]):+.6f}j</b>
</div>

<div class="ii-col">
<b>p₂ = {np.real(poles_pf[1]):.6f} {np.imag(poles_pf[1]):+.6f}j</b><br>
<b>C₂ = {np.real(residues[1]):.6f} {np.imag(residues[1]):+.6f}j</b>
</div>

<div class="ii-col">
<b>p₃ = {np.real(poles_pf[2]):.6f} {np.imag(poles_pf[2]):+.6f}j</b><br>
<b>C₃ = {np.real(residues[2]):.6f} {np.imag(residues[2]):+.6f}j</b>
</div>

</div>

</div>

</div>
"""))

# ============================================================
# DIGITAL TRANSFER FUNCTION
# ============================================================

display(HTML(f"""
<div class="ii-root">

<div class="ii-box ii-note">

<div class="ii-title">Digital transfer function obtained from the analytical procedure</div>

<div class="ii-equation">

<b>
H(z) =
({digital_num[0]:.8f}
{digital_num[1]:+.8f}z<sup>-1</sup>
{digital_num[2]:+.8f}z<sup>-2</sup>)
/
(1
{digital_den[1]:+.8f}z<sup>-1</sup>
{digital_den[2]:+.8f}z<sup>-2</sup>
{digital_den[3]:+.8f}z<sup>-3</sup>)
</b>

</div>

Since the first numerator coefficient is numerically zero,

<div class="ii-equation">

<b>
H(z) =
({digital_num[1]:.8f}z<sup>-1</sup>
{digital_num[2]:+.8f}z<sup>-2</sup>)
/
(1
{digital_den[1]:+.8f}z<sup>-1</sup>
{digital_den[2]:+.8f}z<sup>-2</sup>
{digital_den[3]:+.8f}z<sup>-3</sup>)
</b>

</div>

</div>

</div>
"""))

# ============================================================
# FINAL SOFTWARE VERIFICATION — VISIBLE IN OUTPUT
# ============================================================

display(HTML(f"""
<div class="ii-root">

<div class="ii-box ii-ok">

<div class="ii-title">Final software verification</div>

The complete impulse-invariance design above was obtained explicitly from the
theoretical equations. The following SciPy call is now used <b>only to verify</b>
the final result:

<div class="ii-code">
<b>scipy.signal.cont2discrete((b<sub>a</sub>, a<sub>a</sub>), T<sub>s</sub>, method='impulse')</b>
</div>

<div class="ii-cols">

<div class="ii-col">
<b>Analytical numerator</b><br>
[{manual_num[0]:.10f}, {manual_num[1]:.10f}, {manual_num[2]:.10f}, {manual_num[3]:.10f}]
</div>

<div class="ii-col">
<b>SciPy numerator</b><br>
[{scipy_num[0]:.10f}, {scipy_num[1]:.10f}, {scipy_num[2]:.10f}, {scipy_num[3]:.10f}]
</div>

</div>

<br>

<div class="ii-cols">

<div class="ii-col">
<b>Analytical denominator</b><br>
[{manual_den[0]:.10f}, {manual_den[1]:.10f}, {manual_den[2]:.10f}, {manual_den[3]:.10f}]
</div>

<div class="ii-col">
<b>SciPy denominator</b><br>
[{scipy_den[0]:.10f}, {scipy_den[1]:.10f}, {scipy_den[2]:.10f}, {scipy_den[3]:.10f}]
</div>

</div>

<br>

Maximum numerator difference:
<b>{numerator_difference:.3e}</b>

&nbsp;&nbsp;&nbsp;&nbsp;

Maximum denominator difference:
<b>{denominator_difference:.3e}</b>

<br><br>

<div style="text-align:center;font-size:16px;">
<b>
VERIFICATION:
{"PASSED — the explicit impulse-invariance calculation agrees with SciPy." if verification_passed else "FAILED — the two calculations do not agree within numerical precision."}
</b>
</div>

</div>

</div>
"""))

# ============================================================
# FIGURE — 3 x 2 GRID
# ============================================================

fig,axes = plt.subplots(3,2,figsize=(11.6,11.2))

ax1,ax2,ax3,ax4,ax5,ax6 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# COMMON BOOK-LIKE GRID
# ============================================================

def apply_book_grid(ax):

    ax.grid(True,which='both',linestyle=(0,(1,3)),linewidth=0.8,color='black',alpha=0.85)

    for spine in ax.spines.values():
        spine.set_linewidth(1.0)

# ============================================================
# 1. ANALOG POLES
# ============================================================

ax1.axhline(0,color='black',linewidth=0.8)
ax1.axvline(0,color='black',linewidth=0.8)
ax1.axvspan(-1.0,0.0,alpha=0.05)

ax1.plot(np.real(analog_poles),np.imag(analog_poles),'rx',markersize=8,markeredgewidth=1.8,label='Analog poles')

ax1.set_xlim(-0.8,0.2)
ax1.set_ylim(-0.6,0.6)

ax1.set_title('Analog Poles in the s-Plane')
ax1.set_xlabel(r'$\Re\{s\}$')
ax1.set_ylabel(r'$\Im\{s\}$')

ax1.grid(True,linestyle=':',alpha=0.30)
ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.15),frameon=False)

# ============================================================
# 2. DIGITAL POLES
# ============================================================

theta = np.linspace(0,2*np.pi,1000)

ax2.axhline(0,color='black',linewidth=0.8)
ax2.axvline(0,color='black',linewidth=0.8)

ax2.plot(np.cos(theta),np.sin(theta),'--',linewidth=1.0,label='Unit circle')
ax2.plot(np.real(digital_poles),np.imag(digital_poles),'rx',markersize=8,markeredgewidth=1.8,label='Digital poles')

ax2.set_xlim(-1.15,1.15)
ax2.set_ylim(-1.15,1.15)
ax2.set_aspect('equal',adjustable='box')

ax2.set_title('Digital Poles in the z-Plane')
ax2.set_xlabel(r'$\Re\{z\}$')
ax2.set_ylabel(r'$\Im\{z\}$')

ax2.grid(True,linestyle=':',alpha=0.30)
ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.15),ncol=2,frameon=False)

# ============================================================
# 3. ANALOG MAGNITUDE RESPONSE
# Same frequency range as the textbook figure
# ============================================================

ax3.semilogx(Omega,Ha_dB,color='red',linewidth=1.5)

ax3.set_xlim(1e-2,1e2)
ax3.set_ylim(-140,5)

ax3.set_yticks([0,-20,-40,-60,-80,-100,-120,-140])

ax3.set_title('Magnitude response of the analog filter',fontsize=13.5)
ax3.set_xlabel('Frequency')
ax3.set_ylabel('Logarithmic magnitude (dB)')

apply_book_grid(ax3)

# ============================================================
# 4. ANALOG IMPULSE RESPONSE
# ============================================================

ax4.plot(t,h_analog,color='red',linewidth=1.5)

ax4.set_xlim(0,52)
ax4.set_ylim(-0.05,0.21)

ax4.set_yticks([-0.05,0.00,0.05,0.10,0.15,0.20])

ax4.set_title('Impulse response of the analog filter',fontsize=13.5)
ax4.set_xlabel('Time')
ax4.set_ylabel(r'$h_a(t)$')

apply_book_grid(ax4)

# ============================================================
# 5. DIGITAL MAGNITUDE RESPONSE
# Same normalized-frequency range as the textbook figure
# ============================================================

ax5.plot(omega_normalized,Hd_dB,color='red',linewidth=1.5)

ax5.set_xlim(0,1)
ax5.set_ylim(-60,5)

ax5.set_xticks(np.arange(0,1.01,0.1))
ax5.set_yticks([0,-10,-20,-30,-40,-50,-60])

ax5.set_title('Magnitude response of the digital filter',fontsize=13.5)
ax5.set_xlabel('Frequency')
ax5.set_ylabel('Logarithmic magnitude (dB)')

apply_book_grid(ax5)

# ============================================================
# 6. DIGITAL IMPULSE RESPONSE
# ============================================================

markerline,stemlines,baseline = ax6.stem(n,h_digital,linefmt='r-',markerfmt='ro',basefmt='k-')

plt.setp(markerline,markersize=4.5)
plt.setp(stemlines,linewidth=1.2)
plt.setp(baseline,linewidth=0.9)

ax6.set_xlim(0,52)
ax6.set_ylim(-0.05,0.21)

ax6.set_yticks([-0.05,0.00,0.05,0.10,0.15,0.20])

ax6.set_title('Impulse response of the digital filter',fontsize=13.5)
ax6.set_xlabel('Time')
ax6.set_ylabel(r'$h[n]$')

apply_book_grid(ax6)

# ============================================================
# LAYOUT
# ============================================================

plt.subplots_adjust(left=0.07,right=0.98,top=0.96,bottom=0.06,wspace=0.20,hspace=0.45)

# ============================================================
# DISPLAY
# ============================================================

display(fig.canvas)